# Notebook 03 — Cost-Optimal Threshold tau (RQ3)

Study A. Replace the first paper's arbitrary threshold criterion ("auto-assign
rate >= 80%") with a cost-minimizing optimal threshold tau*.

Expected cost of a threshold policy:
  E[C(tau)] = c_err * integral_{tau}^{1} e(s) f(s) ds
            + c_rev * integral_{0}^{tau} f(s) ds
where e(s) is the SCS-conditional misassignment probability and f(s) the SCS
density. The first-order condition gives
  e(tau*) = c_rev / c_err = 1 / rho,
so the optimum is where the marginal misassignment rate equals the inverse
cost ratio. SCS monotonicity in e(s) guarantees a unique tau* — the point where
the technical index (SCS) and the financial decision (tau*) meet.

Costs are asymmetric: c_err (misassignment / default loss) >> c_rev (manual-
review / opportunity cost). rho = c_err / c_rev sweeps the emerging-market
cost structure.


In [2]:
# %% ============================================================
# Notebook 03 — Cost-Optimal Threshold tau (RQ3)
# Imports, paths, load seller SCS
# ============================================================
import os
import json
import numpy as np
import pandas as pd

SEED = 42
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
ARTIFACT_DIR = os.path.join(ROOT, "artifacts")
TAB_DIR = os.path.join(ROOT, "results", "tables")

scs_df = pd.read_csv(os.path.join(ARTIFACT_DIR, "seller_scs.csv"))
# Use the distribution-based decomposed SCS as the main index (RQ1 winner),
# on the natural-disagreement sellers.
MAIN_SCS = "scs_dist_decomp"
a = scs_df[scs_df["method"] == "A_real"].copy()
a["misassign"] = 1 - a["assign_correct"]
print("Sellers used for tau optimization:", len(a))
print("Overall misassignment rate:", round(a["misassign"].mean(), 4))

# %% ============================================================
# Estimate e(s): SCS-conditional misassignment probability.
# Raw binned estimate is noisy at fine resolution and small samples,
# so we additionally fit an isotonic (monotone non-increasing)
# regression of misassignment on SCS. The isotonic fit gives a
# provably monotone e(s), which is what the tau* uniqueness argument
# requires, while the raw bins are kept for display.
# ============================================================
from sklearn.isotonic import IsotonicRegression

# Raw binned e(s) for display
n_bins = 20
a["scs_bin"] = pd.qcut(a[MAIN_SCS], n_bins, labels=False, duplicates="drop")
ecurve = a.groupby("scs_bin").agg(
    scs_mid=(MAIN_SCS, "mean"),
    e_s=("misassign", "mean"),
    n=("misassign", "size"),
).reset_index()

# Isotonic (monotone non-increasing) fit on seller-level data
iso = IsotonicRegression(increasing=False, out_of_bounds="clip")
iso.fit(a[MAIN_SCS].to_numpy(), a["misassign"].to_numpy())
ecurve["e_s_iso"] = iso.predict(ecurve["scs_mid"].to_numpy())
ecurve.to_csv(os.path.join(ARTIFACT_DIR, "e_of_s_curve.csv"), index=False)

print("e(s) curve (raw bins vs isotonic fit):")
print(ecurve[["scs_mid", "e_s", "e_s_iso", "n"]].round(4).to_string(index=False))

# The isotonic fit is monotone by construction
e_iso_arr = iso.predict(np.sort(a[MAIN_SCS].unique()))
print()
print("raw binned e(s) non-increasing:",
      bool((np.diff(ecurve["e_s"].to_numpy()) <= 1e-9).all()))
print("isotonic e(s) non-increasing:",
      bool((np.diff(e_iso_arr) <= 1e-9).all()))

# %% ============================================================
# Expected cost as a function of tau.
# Sellers with SCS >= tau are auto-assigned; if misassigned they incur
# c_err. Sellers with SCS < tau are sent to manual review, each
# incurring c_rev. We evaluate E[C(tau)] on a tau grid for several
# cost ratios rho = c_err / c_rev.
# ============================================================
def expected_cost(df, scs_col, tau, c_err, c_rev):
    auto = df[df[scs_col] >= tau]
    manual = df[df[scs_col] < tau]
    err_cost = c_err * auto["misassign"].sum()
    rev_cost = c_rev * len(manual)
    return (err_cost + rev_cost) / len(df)


tau_grid = np.linspace(0.0, 1.0, 201)
RHOS = [5, 10, 20, 50, 100]   # emerging-market asymmetric cost ratios
c_rev = 1.0

cost_records = []
for rho in RHOS:
    c_err = rho * c_rev
    costs = [expected_cost(a, MAIN_SCS, t, c_err, c_rev) for t in tau_grid]
    tau_star = tau_grid[int(np.argmin(costs))]
    cost_records.append({"rho": rho, "tau_star": tau_star,
                         "min_cost": min(costs)})
    for t, c in zip(tau_grid, costs):
        cost_records.append({"rho": rho, "tau_grid": t, "cost": c})

tau_star_tbl = pd.DataFrame([r for r in cost_records if "tau_star" in r])[["rho", "tau_star", "min_cost"]]
tau_star_tbl.to_csv(os.path.join(TAB_DIR, "table_tau_star_by_rho.csv"), index=False)
cost_curves = pd.DataFrame([r for r in cost_records if "tau_grid" in r])
cost_curves.to_csv(os.path.join(ARTIFACT_DIR, "tau_cost_curves.csv"), index=False)
print("Optimal tau* by cost ratio rho:")
print(tau_star_tbl.round(4).to_string(index=False))

# %% ============================================================
# First-order-condition check: e(tau*) should approximate 1/rho.
# We read the isotonic (monotone) e(s) at tau* and compare to 1/rho.
# The monotone fit guarantees a unique crossing, hence a unique tau*.
# ============================================================
foc_rows = []
for _, r in tau_star_tbl.iterrows():
    e_at_tau = float(iso.predict([r["tau_star"]])[0])
    foc_rows.append({"rho": int(r["rho"]), "tau_star": r["tau_star"],
                     "e_at_tau_star": e_at_tau, "one_over_rho": 1.0 / r["rho"]})
foc_tbl = pd.DataFrame(foc_rows)
foc_tbl.to_csv(os.path.join(TAB_DIR, "table_tau_foc_check.csv"), index=False)
print("First-order condition: e(tau*) vs 1/rho (isotonic e(s))")
print(foc_tbl.round(4).to_string(index=False))

# %% ============================================================
# Asymmetric-cost sweep: separate false-negative (default) vs
# false-positive (opportunity) error costs. Here c_err represents the
# dominant FN default-loss; we vary the FN/FP ratio to show tau* shift.
# ============================================================
FN_FP = [1, 2, 5, 10]   # c_fn / c_fp ratios
c_fp = 5.0
asym_records = []
for ratio in FN_FP:
    c_fn = ratio * c_fp
    c_err = c_fn
    costs = [expected_cost(a, MAIN_SCS, t, c_err, c_rev) for t in tau_grid]
    tau_star = tau_grid[int(np.argmin(costs))]
    asym_records.append({"c_fn_over_c_fp": ratio, "c_fn": c_fn,
                         "tau_star": tau_star})
asym_tbl = pd.DataFrame(asym_records)
asym_tbl.to_csv(os.path.join(TAB_DIR, "table_tau_asymmetric.csv"), index=False)
print("tau* shift under FN/FP cost asymmetry:")
print(asym_tbl.round(4).to_string(index=False))

# %% ============================================================
# Comparison to the first paper's fixed "80% auto-assign" rule.
# Find the tau that yields an 80% auto-assignment rate, then compare
# its expected cost to the cost-optimal tau* at each rho.
# ============================================================
def auto_rate(df, scs_col, tau):
    return (df[scs_col] >= tau).mean()

rates = np.array([auto_rate(a, MAIN_SCS, t) for t in tau_grid])
idx80 = int(np.argmin(np.abs(rates - 0.80)))
tau_80 = tau_grid[idx80]
print(f"tau for 80% auto-assign rule: {tau_80:.3f} "
      f"(actual rate {rates[idx80]:.3f})")

cmp_rows = []
for rho in RHOS:
    c_err = rho * c_rev
    cost_opt = expected_cost(a, MAIN_SCS,
                             float(tau_star_tbl[tau_star_tbl.rho == rho]["tau_star"].iloc[0]),
                             c_err, c_rev)
    cost_80 = expected_cost(a, MAIN_SCS, tau_80, c_err, c_rev)
    cmp_rows.append({"rho": rho, "cost_optimal": cost_opt,
                     "cost_80rule": cost_80,
                     "savings_pct": 100 * (cost_80 - cost_opt) / cost_80})
cmp_tbl = pd.DataFrame(cmp_rows)
cmp_tbl.to_csv(os.path.join(TAB_DIR, "table_tau_vs_80rule.csv"), index=False)
print()
print("Cost-optimal tau* vs first-paper 80% rule:")
print(cmp_tbl.round(4).to_string(index=False))

Sellers used for tau optimization: 1440
Overall misassignment rate: 0.2833
e(s) curve (raw bins vs isotonic fit):
 scs_mid    e_s  e_s_iso  n
  0.0517 0.4722   0.5271 72
  0.0677 0.5694   0.5271 72
  0.0792 0.5139   0.5271 72
  0.0899 0.5556   0.5271 72
  0.1007 0.4722   0.5106 72
  0.1107 0.4722   0.4400 72
  0.1197 0.4028   0.4231 72
  0.1304 0.3472   0.3540 72
  0.1435 0.3889   0.3540 72
  0.1557 0.2361   0.2778 72
  0.1716 0.2639   0.2478 72
  0.1909 0.2083   0.1646 72
  0.2192 0.1250   0.1646 72
  0.2559 0.1944   0.1646 72
  0.2926 0.0694   0.1148 72
  0.3278 0.1389   0.1148 72
  0.3628 0.0694   0.0719 72
  0.4065 0.0833   0.0719 72
  0.4713 0.0694   0.0612 72
  0.5901 0.0139   0.0000 72

raw binned e(s) non-increasing: False
isotonic e(s) non-increasing: True
Optimal tau* by cost ratio rho:
 rho  tau_star  min_cost
   5      0.19    0.7819
  10      0.33    0.9069
  20      0.52    0.9562
  50      0.52    0.9562
 100      0.52    0.9562
First-order condition: e(tau*) vs 1/rho (i